<a href="https://colab.research.google.com/github/bainiao0706/Google-Drive-Remote-Upload/blob/main/GoogleDriveRemoteUpload.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#谷歌云盘的离线上传
#作者:bainiao0706

In [ ]:
from IPython.display import Javascript
Javascript('window.open("https://github.com/bainiao0706/Google-Drive-Remote-Upload");')

# 挂载Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 离线上传文件

In [ ]:
import requests
import os
from urllib.parse import urlparse, unquote
import threading
import time
import collections
from tqdm.auto import tqdm
import ipywidgets as widgets
from IPython.display import display, HTML
from concurrent.futures import ThreadPoolExecutor, as_completed

!pip install tqdm --quiet # Ensure tqdm is installed

# 下载目录
target_dir = '/content/drive/MyDrive/ColabDownloads' # 配置下载目录，默认下载会直接创建一个ColabDownloads的文件夹，编辑格式:/content/drive/MyDrive/你的文件夹名字

# 初始最大并发下载数量
MAX_CONCURRENT_DOWNLOADS = 5 # 最大并发下载数量，默认32, 已修改为5
DEFAULT_MAX_RETRIES = 3 # 默认最大重试次数, 已修改为3

# 下载文件的函数
def download_file(task, overall_pbar=None): # Added overall_pbar argument
    url = task['url']
    original_filename = task.get('original_filename', '')
    max_retries = task.get('max_retries', DEFAULT_MAX_RETRIES) # 获取任务配置的最大重试次数
    current_max_concurrent_downloads = task.get('current_max_concurrent_downloads', MAX_CONCURRENT_DOWNLOADS) # 获取当前的并发下载数量
    expected_size = task.get('expected_size') # Get expected size from task

    if not url:
        # 如果 URL 为空，返回失败信息
        return {"status": "FAILED", "url": url, "error": "URL is empty.", "retry_attempt": 0, "filename": "Unknown", "expected_size": expected_size}

    # 如果原始文件名为空，则尝试从 URL 中提取
    if not original_filename:
        parsed_url = urlparse(url) # 解析 URL
        filename_from_url = os.path.basename(parsed_url.path) # 获取路径的最后一部分作为文件名
        original_filename = unquote(filename_from_url) # 解码 URL 编码的字符
        if not original_filename:
            original_filename = f"downloaded_file_{threading.get_ident()}.bin" # 如果URL中没有文件名，则使用通用名加线程ID作为备用

    # 使用线程ID来创建临时的唯一文件名，防止不同线程同时下载时文件名冲突
    # 加上原始文件名的basename是为了避免不同URL但相同线程ID的冲突
    temp_filename = f"temp_{threading.get_ident()}_{os.path.basename(original_filename)}.part"
    temp_path = os.path.join(target_dir, temp_filename)
    final_path = os.path.join(target_dir, original_filename)

    for retry_attempt in range(max_retries + 1):
        start_byte = 0
        headers = {}

        if os.path.exists(temp_path):
            start_byte = os.path.getsize(temp_path)
            if start_byte > 0:
                print(f"[任务 {original_filename}] 发现未完成的下载文件 '{temp_filename}'，已下载 {start_byte} 字节。尝试从此处继续...")
                headers = {'Range': f'bytes={start_byte}-'}

        # 修改开始下载的提示信息
        action_msg = "重试下载" if retry_attempt > 0 else "开始下载"
        print(f"[任务 {original_filename}] [尝试 {retry_attempt + 1}] {action_msg}。")

        try:
            # 使用 requests 库下载文件，设置流式传输、允许重定向和超时
            with requests.get(url, stream=True, allow_redirects=True, timeout=60, headers=headers) as r: #请求超时时间默认为60s
                r.raise_for_status()  # 检查 HTTP 请求是否成功 (如 200 OK，否则抛出异常)

                # 确定文件总大小以及是续传还是重新开始
                total_size = 0 # 最终文件的总大小
                current_downloaded_bytes = start_byte # 当前已下载或将从此处开始下载的字节数

                file_mode = 'wb' # 默认以写入模式 (重新开始)

                if r.status_code == 206: # Partial Content - 服务器响应了 Range 头
                    content_range = r.headers.get('Content-Range')
                    if content_range:
                        # 示例: 'bytes 0-999/10000' -> total_size = 10000
                        total_size = int(content_range.split('/')[-1])
                    else: # 如果 Content-Range 缺失但状态是 206，则退而求其次
                        total_size = start_byte + int(r.headers.get('Content-Length', 0))
                    file_mode = 'ab' # 以追加模式写入现有文件
                    print(f"[任务 {original_filename}] 服务器支持断点续传。从 {current_downloaded_bytes} 字节处继续...")
                elif r.status_code == 200: # OK - 服务器发送了完整文件，可能忽略了 Range 或这是一个全新下载
                    total_size = int(r.headers.get('Content-Length', 0))
                    if start_byte > 0: # 如果之前有下载但服务器未响应续传
                        print(f"[任务 {original_filename}] 服务器未响应断点续传请求，将重新开始下载...")
                    current_downloaded_bytes = 0 # 重置初始进度，因为我们重新开始
                    file_mode = 'wb' # 覆盖现有文件

                if total_size == 0: # 如果 Content-Length 或 Content-Range 不可用
                    print(f"[任务 {original_filename}] 警告: 无法获取文件总大小，进度条可能不准确。")
                    total_size = None # tqdm 可以处理 None 作为总大小

                # 使用 tqdm 显示进度条
                # position 参数用于为每个并发下载的进度条设置不同的行，防止重叠
                pos = (threading.get_ident() % current_max_concurrent_downloads)
                with tqdm(
                    initial=current_downloaded_bytes, # 从已下载的字节数开始进度条
                    total=total_size, # 完整文件的总大小
                    unit='B',
                    unit_scale=True,
                    desc=f"下载中 {original_filename}", # 进度条描述
                    miniters=1,
                    smoothing=0.1,
                    position=pos + 2, # Position individual bars below the overall and task completion bars
                    leave=False # 下载完成后不再保留进度条在屏幕上，避免输出混乱
                ) as pbar:
                    with open(temp_path, file_mode) as f: # 使用计算出的模式 'wb' 或 'ab'
                        for chunk in r.iter_content(chunk_size=8192): # 每次读取 8KB 数据块
                            if chunk:
                                f.write(chunk)
                                pbar.update(len(chunk))
                                if overall_pbar: # Update the overall progress bar if provided
                                    overall_pbar.update(len(chunk))

                # 最终检查：如果 total_size 为 None，尝试从最终文件大小获取
                if total_size is None:
                    total_size = os.path.getsize(temp_path)

                os.rename(temp_path, final_path) # 下载完成后，将临时文件重命名为最终文件名
                print(f"[任务 {original_filename}] 成功下载。") # 修改成功下载的提示信息
                # 返回字典格式的成功信息
                return {"status": "SUCCESS", "path": final_path, "url": url, "retry_attempt": retry_attempt, "filename": original_filename, "size": total_size, "expected_size": expected_size}

        except requests.exceptions.RequestException as e:
            if retry_attempt < max_retries:
                sleep_time = 2 ** (retry_attempt + 1) # 指数退避策略: 2s, 4s, 8s, ... (2的n次方秒)
                print(f"[任务 {original_filename}] 下载错误 (尝试 {retry_attempt + 1}/{max_retries + 1}): {e}。将在 {sleep_time} 秒后重试...")
                time.sleep(sleep_time) # 在 worker 线程中休眠，不阻塞 UI
            else:
                print(f"[任务 {original_filename}] 下载失败，已尝试 {retry_attempt + 1} 次。放弃下载。错误: {e}")
                # 返回字典格式的失败信息
                return {"status": "FAILED", "url": url, "error": str(e), "retry_attempt": retry_attempt, "filename": original_filename, "expected_size": expected_size}
        except Exception as e:
            if retry_attempt < max_retries:
                sleep_time = 2 ** (retry_attempt + 1)
                print(f"[任务 {original_filename}] 发生意外错误 (尝试 {retry_attempt + 1}/{max_retries + 1}): {e}。将在 {sleep_time} 秒后重试...")
                time.sleep(sleep_time) # 在 worker 线程中休眠，不阻塞 UI
            else:
                print(f"[任务 {original_filename}] 发生意外错误，已尝试 {retry_attempt + 1} 次。放弃下载。错误: {e}")
                # 返回字典格式的失败信息
                return {"status": "FAILED", "url": url, "error": str(e), "retry_attempt": retry_attempt, "filename": original_filename, "expected_size": expected_size}

    # Should ideally not reach here if max_retries >= 0, but as a fallback
    return {"status": "FAILED", "url": url, "error": "Unknown error or retry limit reached.", "retry_attempt": max_retries, "filename": original_filename, "expected_size": expected_size}

# --- ipywidgets UI 部分 ---

url_input = widgets.Textarea(
    value='',
    placeholder='在此输入下载链接，每行一个URL\n例如: https://example.com/file1.zip\n      https://example.com/file2.rar',
    description='下载URL:',
    disabled=False,
    layout={'width': '80%', 'height': '150px'}
)

download_path_input = widgets.Text(
    value=target_dir,
    placeholder='/content/drive/MyDrive/你的文件夹名字',
    description='保存路径:',
    disabled=False,
    layout={'width': '80%'}
)

max_concurrent_downloads_slider = widgets.IntSlider(
    value=5, # 已修改为5
    min=1,
    max=5, # 交互式UI的最大并发数已改为5
    step=1,
    description='并发下载数量:',
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

max_retries_slider = widgets.IntSlider(
    value=DEFAULT_MAX_RETRIES,
    min=0,
    max=10,
    step=1,
    description='默认重试次数:',
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

download_button = widgets.Button(description='开始下载', button_style='success', icon='cloud-download')
output_widget = widgets.Output()

# 显示 UI 控件
display(url_input, download_path_input, max_concurrent_downloads_slider, max_retries_slider, download_button, output_widget)

# 定义按钮点击事件处理函数
def on_download_button_clicked(b):
    with output_widget:
        output_widget.clear_output() # 清除之前的输出
        print("\n--- 正在启动下载任务 ---")

        global MAX_CONCURRENT_DOWNLOADS
        global DEFAULT_MAX_RETRIES
        global target_dir

        MAX_CONCURRENT_DOWNLOADS = max_concurrent_downloads_slider.value
        DEFAULT_MAX_RETRIES = max_retries_slider.value
        target_dir = download_path_input.value.strip()

        print(f"最大并发下载数量已设置为: {MAX_CONCURRENT_DOWNLOADS}")
        print(f"默认重试次数已设置为: {DEFAULT_MAX_RETRIES}")
        print(f"下载保存路径已设置为: {target_dir}")

        os.makedirs(target_dir, exist_ok=True) # 确保目标目录存在

        # 从 Textarea 获取 URL 列表
        urls = [u.strip() for u in url_input.value.split('\n') if u.strip()]

        if not urls:
            print("错误: 没有提供下载链接。请在文本框中输入URL。")
            return

        # 1. Pre-fetch total sizes for all URLs
        print("正在获取文件大小...")
        total_expected_size = 0
        tasks_to_submit = []
        for u in urls:
            task_info = {
                "url": u,
                "original_filename": "",
                "max_retries": DEFAULT_MAX_RETRIES,
                "current_max_concurrent_downloads": MAX_CONCURRENT_DOWNLOADS,
                "expected_size": None # Default to None
            }
            try:
                # Use HEAD request to get Content-Length
                with requests.head(u, allow_redirects=True, timeout=5) as r_head:
                    r_head.raise_for_status() # Raise HTTPError for bad responses (4xx or 5xx)
                    if 'Content-Length' in r_head.headers:
                        size = int(r_head.headers['Content-Length'])
                        task_info['expected_size'] = size
                        total_expected_size += size
                    else:
                        print(f"警告: URL {u} 的服务器未提供 Content-Length。此文件大小将不计入总进度估算。")
            except requests.exceptions.RequestException as e:
                print(f"警告: 无法获取URL {u} 的大小: {e}。此文件大小将不计入总进度估算。")
            except Exception as e:
                print(f"警告: 处理URL {u} 时发生意外错误: {e}。此文件大小将不计入总进度估算。")
            tasks_to_submit.append(task_info)

        if total_expected_size == 0 and any(task['expected_size'] is not None for task in tasks_to_submit):
            print("注意: 尽管部分文件大小已知，但总预计下载大小为0，这可能表示所有已知文件大小为0或预检失败。")
        elif total_expected_size == 0 and not any(task['expected_size'] is not None for task in tasks_to_submit):
            print("警告: 无法预估任何文件大小，总进度条将仅显示任务完成数量，无法显示预计下载时间或总字节数。")

        if not tasks_to_submit:
            print("没有有效的下载任务。")
            return

        final_results = [] # 存储所有任务的最终结果

        # 2. Create an overall byte progress bar
        # This bar will be updated by individual download_file calls
        overall_byte_pbar = tqdm(total=total_expected_size if total_expected_size > 0 else None,
                                 unit='B', unit_scale=True,
                                 desc=f"总进度 (预计: {total_expected_size / (1024*1024*1024):.2f} GB)" if total_expected_size > 0 else "总进度 (未知大小)",
                                 position=0, leave=True)

        with ThreadPoolExecutor(max_workers=MAX_CONCURRENT_DOWNLOADS) as executor:
            # Submit tasks, passing the overall_byte_pbar to each download_file call
            futures = {executor.submit(download_file, task, overall_byte_pbar): task for task in tasks_to_submit}

            # This tqdm is for monitoring *task completion*, not byte progress directly
            # Position it below the overall byte progress bar
            for future in tqdm(as_completed(futures), total=len(futures), desc="任务完成进度", position=1, leave=False):
                result = future.result()
                final_results.append(result)

        overall_byte_pbar.close() # Close the overall byte progress bar

        print("\n--- 下载总结 ---")
        actual_total_downloaded_bytes = 0
        for res in final_results:
            # 格式化输出结果
            size_info = f"{res.get('size', 'N/A')} 字节" if res.get('size') else 'N/A'
            expected_size_info = f"{res.get('expected_size', 'N/A')} 字节" if res.get('expected_size') is not None else 'N/A'

            print(f"状态: {res['status']}, 文件名: {res.get('filename', 'N/A')}, 大小: {size_info}, 预计大小: {expected_size_info}, URL: {res['url']}")
            if res['status'] == 'FAILED':
                print(f"  错误: {res['error']}")
            else:
                actual_total_downloaded_bytes += res.get('size', 0)

        print(f"\n实际总下载量: {actual_total_downloaded_bytes / (1024*1024*1024):.2f} GB")
        print("------------------------")

# 绑定按钮点击事件
download_button.on_click(on_download_button_clicked)